# 08B_Skill_History_Normalization

Normalize skill names and convert raw counts to adoption rates.

In [36]:
import pandas as pd
import numpy as np

history = pd.read_csv('../Generated Datasets/skill_demand_history.csv')

print(history.shape)
history.head()

(829, 3)


,skill,year,demand
0,.NET,2022,15850
1,.NET (5+),2023,17005
2,.NET (5+),2024,11542
3,.NET Core / .NET 5,2021,15310
4,.NET Framework,2021,16620


## Skill Name Normalization

In [37]:
history['skill'] = (
    history['skill']
    .astype(str)
    .str.strip()
    .str.lower()
)

skill_map = {
    
    'torch/pytorch':'pytorch',

    '.net':'dotnet',
    '.net (5+)':'dotnet',
    '.net core':'dotnet',
    '.net core / .net 5':'dotnet',
    '.net framework':'dotnet',
    '.net framework (1.0 - 4.8)':'dotnet',
    '.net maui':'dotnet',

    'asp.net':'dotnet',
    'asp.net core':'dotnet',

    'c#':'csharp',
    'c sharp':'csharp',

    'c++':'cpp',

    'node.js':'nodejs',
    'node js':'nodejs',

    'postgres':'postgresql',

    'amazon web services':'aws',
    'amazon web services (aws)':'aws',

    'google cloud':'gcp',
    'google cloud platform':'gcp',

    'microsoft azure':'azure',

    'golang':'go',

    'react.js':'react',
    'reactjs':'react',

    'vue.js':'vue',
    'vuejs':'vue',

    'angular.js':'angular',
    'angularjs':'angular',

    'fast api':'fastapi',

    'mongo db':'mongodb',

    'spark':'apache spark',
    'apache spark':'apache spark',

    'power bi':'powerbi',

    'tableau':'tableau',

    'pytorch':'pytorch'
}

history['skill'] = history['skill'].replace(skill_map)

print("Unique Skills:", history['skill'].nunique())

Unique Skills: 264


## Survey Sizes

In [38]:
survey_sizes = {
    2021: 83439,
    2022: 73268,
    2023: 89184,
    2024: 65437,
    2025: 49385
}

history['survey_size'] = history['year'].map(survey_sizes)

history['adoption_rate'] = (
    history['demand']
    / history['survey_size']
)

In [39]:
history_clean = (
    history
    .groupby(['skill','year'], as_index=False)
    [['demand','adoption_rate']]
    .sum()
)

print("Skills:", history_clean['skill'].nunique())
print("Rows:", len(history_clean))

history_clean.head()

Skills: 264
Rows: 810


,skill,year,demand,adoption_rate
0,ada,2023,677,0.007591
1,ada,2024,542,0.008283
2,ada,2025,431,0.008727
3,alibaba cloud,2024,548,0.008374
4,amazon redshift,2025,611,0.012372


In [40]:
history_clean.to_csv(
    '../Generated Datasets/skill_demand_history_clean.csv',
    index=False
)

print("Saved: skill_demand_history_clean.csv")

Saved: skill_demand_history_clean.csv


In [41]:
master = pd.read_csv(
    '../Generated Datasets/master_skill_dataset_v7.csv'
)

master['skill'] = (
    master['skill']
    .astype(str)
    .str.strip()
    .str.lower()
)

master_map = {
    'c#':'csharp',
    'c++':'cpp',
    'node.js':'nodejs',
    'spark':'apache spark',
    'power bi':'powerbi'
}

master['skill'] = master['skill'].replace(master_map)

missing = (
    set(master['skill'])
    - set(history_clean['skill'])
)

print("Missing Skills:")
print(sorted(missing))

Missing Skills:
['langchain', 'llm', 'powerbi', 'rag', 'tableau']
